In [10]:
import torch
import torch.nn as nn 
import torchvision

In [11]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [12]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [13]:
from torchvision.datasets import CIFAR10


In [14]:
train = CIFAR10( root = "./data",train=True,download=True,transform =transform)

100%|██████████████████████████████████████████████████████████████████████████████| 170M/170M [1:07:27<00:00, 42.1kB/s]


In [15]:
test=CIFAR10( root = "./data",train=False,download=True,transform =transform)

In [52]:
from torch.utils.data import DataLoader

trainloader = DataLoader(
    train,
    batch_size=128,
    shuffle=True
)


testloader = DataLoader(
    test,
    batch_size=128,
    shuffle=False
)

In [53]:
import torch
import torch.nn as nn

class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
              nn.Linear(4 * 4 * 128, 256),
                 nn.ReLU(),
                nn.Linear(256, 10)
         )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_layers(x)
        return x

In [54]:
#building model
model = CNN()

In [55]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [57]:
# Training the CNN

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)  # Forward propagation

        loss = criterion(output, labels)  # Calculate loss

        loss.backward()  # Backpropagation

        optimizer.step()  # Update parameters

        epoch_training_loss += loss.item()

    print(f"epoch={epoch + 1} & loss={epoch_training_loss / len(trainloader):.4f}")

epoch=1 & loss=1.4528
epoch=2 & loss=1.0252
epoch=3 & loss=0.8430
epoch=4 & loss=0.7198
epoch=5 & loss=0.6165
epoch=6 & loss=0.5358
epoch=7 & loss=0.4535
epoch=8 & loss=0.3795
epoch=9 & loss=0.3091
epoch=10 & loss=0.2455


In [58]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.11
